In [3]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

2.8.0+cpu
CUDA available: False


In [1]:
import torch, torchvision, platform
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("torch wheel CUDA:", torch.version.cuda)         # e.g., '12.1' for cu121 wheels
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device count:", torch.cuda.device_count())
    print("Device name:", torch.cuda.get_device_name(0))
    print("cuDNN version:", torch.backends.cudnn.version())

torch: 2.5.1+cu121
torchvision: 0.20.1+cu121
torch wheel CUDA: 12.1
CUDA available: True
Device count: 1
Device name: NVIDIA GeForce GTX 1650 with Max-Q Design
cuDNN version: 90100


In [ ]:
from huggingface_hub import notebook_login

# Prompt for Hugging Face token in the notebook UI
notebook_login()

In [4]:
from PIL import Image
import numpy as np
import requests
import torch

from transformers import DPTImageProcessor, DPTForDepthEstimation

image_processor = DPTImageProcessor.from_pretrained("Intel/dpt-hybrid-midas")
model = DPTForDepthEstimation.from_pretrained("Intel/dpt-hybrid-midas", low_cpu_mem_usage=True)

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# prepare image for the model
inputs = image_processor(images=image, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    predicted_depth = outputs.predicted_depth

# interpolate to original size
prediction = torch.nn.functional.interpolate(
    predicted_depth.unsqueeze(1),
    size=image.size[::-1],
    mode="bicubic",
    align_corners=False,
)

# visualize the prediction
output = prediction.squeeze().cpu().numpy()
formatted = (output * 255 / np.max(output)).astype("uint8")
depth = Image.fromarray(formatted)
depth.show()


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434